In [177]:
import pandas as pd
import numpy as np

df_transacoes = pd.read_csv("Base_de_Transacoes_e_Cupons_Capturados.csv", sep=";")
df_simulacao = pd.read_csv("Base_Simulada_-_Pedestres_Av__Paulista.csv", sep=";")

df_transacoes = df_transacoes[["celular", "hora", "data"]].reset_index(drop=True)
df_transacoes["data"] = pd.to_datetime(df_transacoes["data"], dayfirst=True, errors="coerce")
df_transacoes["hora"] = pd.to_timedelta(df_transacoes["hora"].astype(str), errors="coerce")

df_simulacao = df_simulacao[df_simulacao["possui_app_picmoney"] == "Sim"]
df_simulacao = df_simulacao[["celular", "horario", "data"]].rename(columns={"horario":"hora"}).reset_index(drop=True)

df = pd.concat([df_transacoes, df_simulacao])

In [178]:
transacoes_totais = len(df)

usuarios_unicos = len(df["celular"].unique())

dias_analisados = len(df["data"].unique())

print(transacoes_totais)
print(usuarios_unicos)
print(dias_analisados)

159957
64745
32


In [179]:
DAU = df_transacoes.groupby("data")["celular"].nunique()
print(DAU)
DAU = DAU.mean()
DAU = round(DAU,2)
print(f"DAU = {DAU}")

data
2025-07-01    2181
2025-07-02    2160
2025-07-03    2163
2025-07-04    2175
2025-07-05    2203
2025-07-06    2193
2025-07-07    2195
2025-07-08    2102
2025-07-09    2155
2025-07-10    2152
2025-07-11    2186
2025-07-12    2203
2025-07-13    2194
2025-07-14    2119
2025-07-15    2161
2025-07-16    2109
2025-07-17    2153
2025-07-18    2124
2025-07-19    2120
2025-07-20    2136
2025-07-21    2194
2025-07-22    2164
2025-07-23    2186
2025-07-24    2145
2025-07-25    2168
2025-07-26    2191
2025-07-27    2129
2025-07-28    2138
2025-07-29    2132
2025-07-30    2120
2025-07-31    2148
Name: celular, dtype: int64
DAU = 2158.03


In [180]:
WAU = df_transacoes.groupby(df_transacoes["data"].dt.isocalendar().week)["celular"].nunique()
print(WAU)
WAU = WAU.mean()
WAU = round(WAU, 2)
print(f"DAU = {WAU}")


week
27    4416
28    4514
29    4483
30    4510
31    4003
Name: celular, dtype: int64
DAU = 4385.2


In [181]:
MAU = df_transacoes.groupby(df_transacoes["data"].dt.month)["celular"].nunique()
print(MAU)
MAU = MAU.mean()
MAU = round(MAU, 2)
print(f"DAU = {MAU}")

data
7    4813
Name: celular, dtype: int64
DAU = 4813.0


In [182]:
data_minima = df_transacoes["data"].min()
data_limite = data_minima + pd.Timedelta(days=6)
data_retorno = data_minima + pd.Timedelta(days=30)

usuarios_iniciais = df_transacoes[df_transacoes["data"].between(data_minima, data_limite)]["celular"].unique()
usuarios_futuros = df_transacoes[df_transacoes["data"] >= data_retorno]["celular"].unique()

usuarios_retidos = set(usuarios_iniciais) & set(usuarios_futuros)
taxa_retencao = len(usuarios_retidos) / len(usuarios_iniciais)
taxa_retencao = taxa_retencao*100
taxa_retencao = round(taxa_retencao, 2)
taxa_retencao


46.2

In [183]:
end_period = pd.Timestamp(year=df_transacoes["data"].max().year, month=7, day=31)

dfp = df_transacoes[df_transacoes["data"] <= end_period]
last_tx = dfp.groupby("celular")["data"].max().rename("ultima")
inatividade = (end_period - last_tx).dt.days.clip(lower=0)

usuarios_total = last_tx.shape[0]
usuarios_inativos = (inatividade >= 14).sum()
taxa_parada = usuarios_inativos / usuarios_total
taxa_parada = round(taxa_parada*100, 2)

bins = [0, 7, 14, 21, 31]
labels = ["0-7", "8-14", "15-21", "22-31"]
faixas = pd.cut(inatividade.clip(upper=31), bins=bins, labels=labels, include_lowest=True, right=True)
dist = faixas.value_counts().reindex(labels, fill_value=0).rename("qtd").to_frame()
dist["percentual"] = (dist["qtd"] / usuarios_total * 100).round(2)

print(f" Usuários inativos = {usuarios_inativos}")
print(f"usuarios totais = {usuarios_total}")
print(f"taxa churn = {taxa_parada}") 
dist.reset_index(names="faixa")

 Usuários inativos = 61
usuarios totais = 4813
taxa churn = 1.27


,faixa,qtd,percentual
0,0-7,4567,94.89
1,8-14,194,4.03
2,15-21,33,0.69
3,22-31,19,0.39


In [184]:

df_transacoes["timestamp"] = df_transacoes["data"].dt.normalize() + df_transacoes["hora"]
df_transacoes = df_transacoes.sort_values(["celular", "timestamp"]).copy()

df_transacoes["gap_horas"] = df_transacoes.groupby("celular")["timestamp"].diff().dt.total_seconds() / 3600
df_transacoes["nova_sessao"] = df_transacoes["gap_horas"].isna() | (df_transacoes["gap_horas"] > 4)
df_transacoes["session_id"] = df_transacoes.groupby("celular")["nova_sessao"].cumsum()

sessoes = (
    df_transacoes.groupby(["celular", "session_id"])["timestamp"]
      .agg(inicio="min", fim="max", transacoes="count")
      .reset_index()
)
sessoes["duracao_horas"] = (sessoes["fim"] - sessoes["inicio"]).dt.total_seconds() / 3600

sessoes_multiplas = sessoes[sessoes["transacoes"] > 1].shape[0]
media_duracao = round(sessoes["duracao_horas"].mean() * 60, 2)

bins = [0, 0.25, 0.5, 1, 2, 4, float("inf")]
labels = ["0-15 min", "15-30 min", "30-60 min", "1-2 h", "2-4 h", "4+ h"]
sessoes["faixa"] = pd.cut(sessoes["duracao_horas"], bins=bins, labels=labels, include_lowest=True, right=True)

dist = sessoes["faixa"].value_counts().reindex(labels, fill_value=0).rename("qtd").to_frame()
dist["percentual"] = (dist["qtd"] / sessoes.shape[0] * 100).round(2)

print(f"numero de sessoes multiplas = {sessoes_multiplas}")
print(f"media de durção = {media_duracao}") 
dist.reset_index(names="faixa")

numero de sessoes multiplas = 14154
media de durção = 24.48


,faixa,qtd,percentual
0,0-15 min,69146,84.32
1,15-30 min,834,1.02
2,30-60 min,1669,2.04
3,1-2 h,3158,3.85
4,2-4 h,5769,7.04
5,4+ h,1425,1.74


In [185]:
df_transacoes["timestamp"] = df_transacoes["data"].dt.normalize() + df_transacoes["hora"]
df_transacoes = df_transacoes.sort_values(["celular", "timestamp"]).copy()

df_transacoes["gap_horas"] = df_transacoes.groupby("celular")["timestamp"].diff().dt.total_seconds() / 3600
df_transacoes["nova_sessao"] = df_transacoes["gap_horas"].isna() | (df_transacoes["gap_horas"] > 4)
df_transacoes["session_id"] = df_transacoes.groupby("celular")["nova_sessao"].cumsum()

sessoes_por_usuario = df_transacoes.groupby("celular")["session_id"].nunique().reset_index(name="qtd_sessoes")

total_sessoes = sessoes_por_usuario["qtd_sessoes"].sum()
total_usuarios = sessoes_por_usuario.shape[0]
media_sessoes = round(total_sessoes / total_usuarios, 2)
top5_usuarios = sessoes_por_usuario.sort_values("qtd_sessoes", ascending=False).head(5)

print(f"{total_sessoes}") 
print(f"{total_usuarios}") 
print(f"{media_sessoes}")
top5_usuarios

82001
4813
17.04


,celular,qtd_sessoes
254,(11) 91538-7360,54
901,(11) 92941-5524,52
1868,(11) 94969-8171,50
3861,(11) 99054-6824,48
3951,(11) 99209-7919,48


In [186]:
resumo_executivo = pd.DataFrame({
    "Indicador": [
        "Usuários Ativos Diários (DAU)",
        "Usuários Ativos Semanais (WAU)",
        "Usuários Ativos Mensais (MAU)",
        "Taxa de Retenção (30 dias)",
        "Taxa de Parada (Churn)",
        "Tempo Médio na Aplicação (min)",
        "Sessões por Usuário"
    ],
    "Valor": [
        DAU,
        WAU,
        MAU,
        f"{taxa_retencao}%",
        f"{taxa_parada}%",
        f"{media_duracao} min",
        f"{media_sessoes} sessões/usuario"
    ]
})

resumo_executivo


,Indicador,Valor
0,Usuários Ativos Diários (DAU),2158.03
1,Usuários Ativos Semanais (WAU),4385.2
2,Usuários Ativos Mensais (MAU),4813.0
3,Taxa de Retenção (30 dias),46.2%
4,Taxa de Parada (Churn),1.27%
5,Tempo Médio na Aplicação (min),24.48 min
6,Sessões por Usuário,17.04 sessões/usuario
